# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHIT-25607/FLYRANK-INTERN/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

I audited the FlyRank research paper (`docs/flyrank-seo-research-march-2026.pdf`) first, and the audit gave me
three questions that generalize to my own Week-5 model. This notebook applies them to myself:

1. Where does the **label** come from — and is the validation design built to carry the claim?
2. Does the **split** respect the data's real structure (whole clients, not random pages)?
3. What would a **leak** actually do to my numbers if one slipped in?

Sections 1-3 are the audit; section 4 rewrites my own boldest sentence in safe language. The w04/w05 numbers I
am re-auditing are in `work/outputs/w04_baseline_metrics.json` and `work/outputs/w05_model_metrics.json`.

## 1. Two paper findings + my methodology questions

The paper's evidence standard is genuinely careful — "direct aggregate comparisons lead", ML pages are
"exploratory appendix material", weaker internal constructs get demoted (pp. 4-5). My questions are about the
parts it does *not* print, which is exactly what my own notebook should print.

**Finding A — "Growing content is longer, younger, and positioned slightly better."** *(Finding #1, CONFIRMED)*

- **As printed:** rising pages average 3.2K vs 2.3K words (37.6% longer) and 184 vs 230 days old, over large
  cohorts (74.2K rising vs 45.3K falling).
- **Where does the label come from?** `trend_direction` is a **same-window** 30d-vs-prev-30d impression change
  (p.5: Up > +10%, Down > -10%). The up/down label and the profiling features (words, age, position, health) live
  in the **same window** — no time gap, so this is a descriptive cross-section, not prediction.
- **Does the validation carry the claim?** For the descriptive profile, yes — it is honestly called an
  observational comparison (p.6) and the evidence standard demotes weaker constructs. What it cannot carry is the
  "What to Do With This" advice (expand thin pages, refresh quarterly): a same-window contrast cannot prove that
  refreshing *causes* retention.
- **My questions:** (1) Would the profile survive if "growing" meant *next*-period growth? (2) Counts are shown
  but no base rate — what share of the portfolio was up vs down? (3) Position is embedded in the health metric
  in the same panel, so part of the "position" difference is the metric colliding with itself.

**How I answered in my lane.** My label is `declined_next_30d = (apr_imp < 0.8 * imp_march)` — an April (future)
observation against March-only features. The window trap the paper's trend label sits on is structurally absent
from mine. Cell 1a below proves the windows are adjacent; 1b prints the base rates the paper omits; 1c shows the
split difference before we even train.

**Finding B — "Logistic regression: 71% holdout accuracy predicting growing vs declining pages."** *(ML appendix,
Growth & Classification)*

- **As printed:** 71% holdout accuracy (p.29); the same appendix reports Random Forest "feature importance for
  health score" with Average Position 43% and Impressions 32% (p.27).
- **Where does the label come from?** The same **same-window** `trend_direction` buckets again — label and
  features share one window.
- **Does the validation carry the claim?** Four things the appendix does not print: **(1) base rate** — accuracy
  is unreadable on an imbalanced label; a "predict the majority class" rule can score 60-71% with zero signal.
  **(2) grouping** — 57 brands in the portfolio; a plain random holdout puts the same brand's pages on both
  sides, letting the model memorize brand-level noise. **(3) shared construct** — the health-score importance is
  partly mechanical, because health = impressions (30) + position (30) + CTR (20) + scroll (20); the paper itself
  flags it ("partly constructed from... these inputs"). **(4) metric choice** — accuracy picks a threshold;
  a ranking task is better read with precision@K + ROC-AUC.
- **My questions, condensed:** what base rate, what kind of holdout, and which columns feed the label?

**How I answered in my lane.** P@K + ROC-AUC with base rates printed for train and held-out; whole clients held
out (30 train / 13 held-out, overlap 0); permutation importance measured on held-out pages; label never derives
from the features. The rest of this notebook re-runs my own w05 model under exactly those three questions.

In [1]:
# Section 1 back-up numbers: window separation, the accuracy trap, both split designs.
import os, json, duckdb, pandas as pd, numpy as np
from pathlib import Path

root = Path(".").resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
OUT = root / "work" / "outputs"
CACHE = root / "work" / "outputs" / "w03_cache"

import sklearn, sys
print("python", sys.version.split()[0], "| duckdb", duckdb.__version__,
      "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

def hf_token():
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not tok:
        import getpass
        tok = getpass.getpass("HF_TOKEN: ")
    return tok

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + hf_token() + "')")
con.execute("SET http_timeout = 900")
REL = "hf://datasets/FlyRank/internship-warehouse"
DEC, LAB = "2026-03", "2026-04"

# Same lane build as w03..w05. Read the local month cache when present (fast, no
# network); otherwise fall back to the warehouse partition. Query shape is identical.
def month_path(month):
    name = {"2026-03": "fact_2026-03.parquet", "2026-04": "fact_2026-04.parquet"}[month]
    p = CACHE / name
    if p.exists():
        return f"read_parquet('{p.as_posix()}')"
    return f"read_parquet('{REL}/fact_content_daily_performance/month={month}/data_0.parquet')"

ff = con.sql(f'''
    WITH dec AS (
      SELECT * FROM {month_path(DEC)} WHERE gsc_data_available IS TRUE
    )
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_march,
           SUM(gsc_clicks)      AS clk_march,
           AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS pos_march,
           COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS active_days_march,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-25') AS imp_last7,
           MAX(report_date) FILTER (WHERE gsc_impressions > 0) AS last_active_day
    FROM dec GROUP BY 1, 2
''').df()

lab = con.sql(f'''
    SELECT content_hash_id, SUM(gsc_impressions) AS apr_imp
    FROM {month_path(LAB)}
    WHERE gsc_data_available IS TRUE GROUP BY 1
''').df()

DEC_END = pd.Timestamp("2026-03-31")
df = ff.merge(lab, on="content_hash_id", how="inner")
df = df[df["imp_march"] >= 100].reset_index(drop=True)
df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
df["declined_next_30d"] = (df["apr_imp"] < 0.8 * df["imp_march"]).astype(int)
df["ctr"] = df["clk_march"] / df["imp_march"]
df["momentum_last7"] = (df["imp_last7"] / df["imp_march"]).fillna(0.0)
df["inert_days"] = (DEC_END - pd.to_datetime(df["last_active_day"])).dt.days.fillna(30).astype(float)

def pos_band(p):
    if pd.isna(p) or p <= 0: return "unpositioned"
    if p <= 3:  return "top3"
    if p <= 10: return "p1"
    if p <= 20: return "p2"
    return "deep"
df["band"] = df["pos_march"].map(pos_band)
df["pos_valid"] = df["pos_march"].notna().astype(int)
base = round(float(df["declined_next_30d"].mean()), 3)
print("lane pool :", len(df), "pages |", df["client_hash_id"].nunique(), "clients | base", base)

print("\n--- 1a. window separation (the paper's trend label is same-window; mine is adjacent future) ---")
feat_dates = con.sql(f"SELECT MIN(report_date), MAX(report_date) FROM {month_path(DEC)} WHERE gsc_data_available IS TRUE").fetchone()
lab_dates  = con.sql(f"SELECT MIN(report_date), MAX(report_date) FROM {month_path(LAB)} WHERE gsc_data_available IS TRUE").fetchone()
print("feature window:", feat_dates[0], "->", feat_dates[1])
print("label window  :", lab_dates[0], "->", lab_dates[1])
assert lab_dates[0] > feat_dates[1], "label window must start AFTER the feature window closes"
print("adjacent, non-overlapping -> the decline label is a genuine prior-to-future outcome.")

print("\n--- 1b. the accuracy trap: majority-class rules already 'score' on an imbalanced slice ---")
print("whole pool decline base:", base)
print("majority-class 'accuracy' (predict 'declined' for everyone):", base)
print("majority-class 'accuracy' (predict 'stable'  for everyone):", round(1 - base, 3))
print("=> an accuracy-only headline (e.g. '71% holdout accuracy') is unreadable without the")
print("   class mix behind it. My lane prints P@K + ROC-AUC next to base rates instead.")

print("\n--- 1c. the split designs the paper could be using vs mine ---")
FEATS = ["imp_march", "clk_march", "ctr", "pos_march", "active_days_march",
         "momentum_last7", "inert_days", "band", "pos_valid"]
X = df[FEATS + ["declined_next_30d", "client_hash_id"]].copy()
Xnum = X.select_dtypes(include=[np.number]).drop(columns=["declined_next_30d"])
Xnum = Xnum.fillna({"pos_march": 0.0, "momentum_last7": 0.0})
from sklearn.preprocessing import OrdinalEncoder
band_enc = OrdinalEncoder(dtype=float).fit_transform(X[["band"]].to_numpy().reshape(-1, 1)).ravel()
X_model = Xnum.assign(band=band_enc.astype(float)).astype(float)
y = X["declined_next_30d"].astype(int).reset_index(drop=True)
groups = X["client_hash_id"].reset_index(drop=True)

from sklearn.model_selection import GroupShuffleSplit, train_test_split
tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
                  .split(X_model, y, groups))
rt, rt2 = train_test_split(np.arange(len(y)), test_size=0.30, random_state=42, stratify=y.to_numpy())
ov_rand = len(set(groups.iloc[rt]) & set(groups.iloc[rt2]))
ov_grp  = len(set(groups.iloc[tr_i]) & set(groups.iloc[te_i]))
print("random split (naive)  : held-out pages come from", ov_rand, "of",
      groups.iloc[rt2].nunique(), "clients that also sit in train")
print("grouped split (mine)  : held-out clients seen in train:", ov_grp, "of",
      groups.iloc[te_i].nunique())
print("features:", FEATS)


python 3.12.0 | duckdb 1.5.5 | pandas 2.2.3 | sklearn 1.6.1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

lane pool : 100893 pages | 43 clients | base 0.515

--- 1a. window separation (the paper's trend label is same-window; mine is adjacent future) ---


feature window: 2026-03-01 00:00:00 -> 2026-03-31 00:00:00
label window  : 2026-04-01 00:00:00 -> 2026-04-30 00:00:00
adjacent, non-overlapping -> the decline label is a genuine prior-to-future outcome.

--- 1b. the accuracy trap: majority-class rules already 'score' on an imbalanced slice ---
whole pool decline base: 0.515
majority-class 'accuracy' (predict 'declined' for everyone): 0.515
majority-class 'accuracy' (predict 'stable'  for everyone): 0.485
=> an accuracy-only headline (e.g. '71% holdout accuracy') is unreadable without the
   class mix behind it. My lane prints P@K + ROC-AUC next to base rates instead.

--- 1c. the split designs the paper could be using vs mine ---


random split (naive)  : held-out pages come from 39 of 39 clients that also sit in train
grouped split (mine)  : held-out clients seen in train: 0 of 13
features: ['imp_march', 'clk_march', 'ctr', 'pos_march', 'active_days_march', 'momentum_last7', 'inert_days', 'band', 'pos_valid']


## 2. My model under an honest split (before/after)

Same data, same features, same Random Forest (400 trees / `min_samples_leaf=15` / `sqrt` / seed 42). **Only the
split changes:**

| split | what it does | why it is on this list |
|---|---|---|
| **BEFORE — random 70/30, stratified** | test pages come from the same clients that trained the model (overlap 39/39 here) | the naive default, and the plausible reading of the paper's "holdout accuracy" |
| **AFTER — grouped by client** (30 train / 13 held-out) | a whole set of clients is never seen | simulates "score a NEW client's pages" — this is what w05 shipped |

w05 kept this same grouped split for the rule, LR and RF. Here I re-fit only the RF because the question is
"does the split change the answer?" — and I add an after-after GroupKFold so one lucky 30/13 draw cannot carry
the conclusion.

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def run_rf(tr, te):
    rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=15, max_features="sqrt",
                                random_state=42, n_jobs=-1).fit(X_model.iloc[tr], y.iloc[tr])
    s = rf.predict_proba(X_model.iloc[te])[:, 1]
    yte = y.iloc[te].to_numpy()
    order = np.argsort(-s)
    pk = {k: round(float(yte[order[:k]].mean()), 3) for k in [10, 20, 50, 100]}
    auc = round(float(roc_auc_score(yte, s)), 3)
    tr_auc = round(float(roc_auc_score(y.iloc[tr], rf.predict_proba(X_model.iloc[tr])[:, 1])), 3)
    return pk, auc, tr_auc, yte

k_hdr = "".join(f" P@{k:<6}" for k in [10, 20, 50, 100])
print(f"{'split':<24}" + k_hdr + f"  AUC  [tr AUC]  base   overlap")
print("-" * 78)
results = {}
for name, tr, te in [("BEFORE  random 70/30", rt, rt2),
                     ("AFTER   grouped by client", tr_i, te_i)]:
    pk, auc, tr_auc, yte = run_rf(tr, te)
    ov = len(set(groups.iloc[tr]) & set(groups.iloc[te]))
    results[name] = {"pk": pk, "auc": auc, "tr_auc": tr_auc, "base": round(float(yte.mean()), 3), "overlap": ov}
    print(f"{name:<24}" + "".join(f" {pk[k]:<8.3f}" for k in [10, 20, 50, 100]) +
          f"  {auc:.3f}  [{tr_auc:.3f}]  {yte.mean():.3f}  {ov}")

before, after = results["BEFORE  random 70/30"], results["AFTER   grouped by client"]
print("\nRead: the random split trains on 39 of the same 39 test clients, so the model has already")
print("met those pages' history. The grouped split keeps overlap at 0. Which number is 'the result'")
print("depends entirely on which question you are asking - new-client scoring wants the grouped one.")

from sklearn.model_selection import GroupKFold
p50s, aucs = [], []
for tr, te in GroupKFold(n_splits=5).split(X_model, y, groups):
    pk, auc, _, _ = run_rf(tr, te)
    p50s.append(pk[50]); aucs.append(auc)
print("\nAFTER-AFTER: GroupKFold (5 folds, whole clients held out each fold)")
print("  P@50 mean", round(float(np.mean(p50s)), 3), "+/-", round(float(np.std(p50s)), 3))
print("  AUC  mean", round(float(np.mean(aucs)), 3), "+/-", round(float(np.std(aucs)), 3))
print("  every fold: overlap 0 by construction - the single 30/13 split was not a lucky draw.")


split                    P@10     P@20     P@50     P@100     AUC  [tr AUC]  base   overlap
------------------------------------------------------------------------------


BEFORE  random 70/30     0.900    0.950    0.920    0.960     0.770  [0.853]  0.515  39


AFTER   grouped by client 0.900    0.900    0.860    0.870     0.739  [0.852]  0.429  0

Read: the random split trains on 39 of the same 39 test clients, so the model has already
met those pages' history. The grouped split keeps overlap at 0. Which number is 'the result'
depends entirely on which question you are asking - new-client scoring wants the grouped one.



AFTER-AFTER: GroupKFold (5 folds, whole clients held out each fold)
  P@50 mean 0.944 +/- 0.023
  AUC  mean 0.751 +/- 0.024
  every fold: overlap 0 by construction - the single 30/13 split was not a lucky draw.


## 3. Leakage audit

The same hunt from Week 3, on the **final** feature set. Two leak families:

1. **Time window** — every feature must end on 2026-03-31. One tempting dimension is deliberately rejected:
   staleness from `dim_content.content_updated_date` is an *as-of-release* snapshot (many rows carry
   2026-07-01 dates), so it is NOT knowable at the decision moment — it would smuggle in the future.
2. **Label reach** — `declined_next_30d` and its source `apr_imp` must never ride along as features.

The cell asserts all of it, then proves the trap is real: adding April impressions to the matrix shows exactly the
score jump an accidental leak would have bought.

In [3]:
print("--- knowability of every shipped feature at the 2026-03-31 decision moment ---")
know = [
    ("imp_march",           "sum of March impressions",                    "Yes - March partition"),
    ("clk_march",           "sum of March clicks",                        "Yes - March partition"),
    ("ctr",                 "clk_march / imp_march",                      "Yes - March only"),
    ("pos_march",           "avg March position",                         "Yes - March only"),
    ("active_days_march",   "days with impressions in March",             "Yes - March only"),
    ("momentum_last7",      "impressions Mar 25-31 / March total",        "Yes - Mar 31 fully closed"),
    ("inert_days",          "days since last active day",                 "Yes - capped at Mar 31"),
    ("band / pos_valid",    "position band + validity",                   "Yes - derived from March pos"),
    ("content_updated_date","dim_content staleness (REJECTED)",           "NO - as-of-release cloud 2026-07 dates"),
    ("apr_imp",             "label-month impressions (REJECTED)",         "NO - April window"),
]
for f, what, avail in know:
    print(f"  {f:<24}{what:<36}{avail}")
for f, _, avail in know:
    if f == "apr_imp": continue
assert all("Yes" in a for _, _, a in know[:8]), "a shipped feature is not knowable at the decision moment"
print("\nasserted: all 8 shipped features are knowable by 2026-03-31; the two rejected ones are not.")

assert "declined_next_30d" not in FEATS
assert "apr_imp" not in FEATS and "apr_imp" not in X_model.columns
assert not set(X_model.columns) & {"apr_imp", "declined_next_30d"}
print("asserted: the label and its April source are not in the model matrix.")

assert ov_grp == 0
print("asserted: the grouped split keeps whole clients apart (overlap =", ov_grp, ").")

print("\n--- leak trap: add April impressions as a feature, same split, re-train ---")
leak = X_model.assign(apr_imp=np.log1p(df["apr_imp"].to_numpy()))
lf = RandomForestClassifier(n_estimators=400, min_samples_leaf=15, max_features="sqrt",
                            random_state=42, n_jobs=-1).fit(leak.iloc[tr_i], y.iloc[tr_i])
ls = lf.predict_proba(leak.iloc[te_i])[:, 1]
yte = y.iloc[te_i].to_numpy()
l_p50 = round(float(yte[np.argsort(-ls)[:50]].mean()), 3)
l_auc = round(float(roc_auc_score(yte, ls)), 3)
print(f"leak trap (apr_imp in features): P@50 {l_p50} | AUC {l_auc}")
print(f"honest    (shipped feature set): P@50 {after['pk'][50]} | AUC {after['auc']}")
print("=> a real future-leak pays off visibly. The shipped set has no such column; the gap")
print("   train->test (%.3f -> %.3f) is the normal tree-tightening, not leak: overlap is" % (after["tr_auc"], after["auc"]))
print("   0 clients and no feature reaches past Mar 31.")


--- knowability of every shipped feature at the 2026-03-31 decision moment ---
  imp_march               sum of March impressions            Yes - March partition
  clk_march               sum of March clicks                 Yes - March partition
  ctr                     clk_march / imp_march               Yes - March only
  pos_march               avg March position                  Yes - March only
  active_days_march       days with impressions in March      Yes - March only
  momentum_last7          impressions Mar 25-31 / March total Yes - Mar 31 fully closed
  inert_days              days since last active day          Yes - capped at Mar 31
  band / pos_valid        position band + validity            Yes - derived from March pos
  content_updated_date    dim_content staleness (REJECTED)    NO - as-of-release cloud 2026-07 dates
  apr_imp                 label-month impressions (REJECTED)  NO - April window

asserted: all 8 shipped features are knowable by 2026-03-31; the two r

leak trap (apr_imp in features): P@50 1.0 | AUC 0.995
honest    (shipped feature set): P@50 0.86 | AUC 0.739
=> a real future-leak pays off visibly. The shipped set has no such column; the gap
   train->test (0.852 -> 0.739) is the normal tree-tightening, not leak: overlap is
   0 clients and no feature reaches past Mar 31.


## 4. Claim rewrite

Reading my w05 error-and-interpretation section back with the paper-audit glasses on, the bold sentences were in
the family of "the model *is* good and *leans* on momentum". Fine for a working notebook, too loose as a submitted
claim. Rewrites below follow one rule the paper taught me: **observed, measured, directional, decision-support** —
never causal, never global, and never a forecast.

In [4]:
rewrites = [
    ("P@50 0.86 / P@100 0.87 / AUC 0.739 on held-out clients - far above the baseline (0.44).",
     "Measured on the 13 held-out clients (base rate 0.429), the random forest scored P@50 0.86, P@100 0.87 and "
     "ROC-AUC 0.739 against the April-decline label, while the w04 rule scored P@50 0.44 on the same pages. "
     "Observed on this one client slice - not yet proven to generalize to other client mixes."),
    ("momentum_last7 dominates the model's importance.",
     "In this sample, permutation importance measured on held-out pages ranked momentum_last7 first. That is "
     "directional evidence that recent-momentum loss co-occurs with the decline label - an association, not a "
     "proven cause, and not a lever you can pull to force retention."),
    ("The two rankings share no top-50 pages, so both notions of 'worth reviewing' are real.",
     "On this slice the rule and the model share zero top-50 pages and both clear the rule's own P@K on the "
     "same split. That is an observation about two different ranking semantics, read as complementary "
     "decision-support inputs - not a verdict on which ranking is correct."),
]
for orig, safe in rewrites:
    print("ORIG:", orig)
    print("SAFE:", safe)
    print()

metrics = {"task": "ml-09", "decision_month": DEC, "label_month": LAB,
           "source": "re-audit of w05_model_metrics.json + w04_baseline_metrics.json",
           "section1": {
               "feature_window": [str(feat_dates[0]), str(feat_dates[1])],
               "label_window":   [str(lab_dates[0]), str(lab_dates[1])],
               "adjacent_non_overlapping": bool(lab_dates[0] > feat_dates[1]),
               "whole_pool_base_rate": base,
               "majority_class_accuracy_decline": base,
               "majority_class_accuracy_stable": round(1 - base, 3),
               "random_split_client_overlap": ov_rand,
               "grouped_split_client_overlap": ov_grp},
           "section2": {
               "before_random_70_30": {"pk": before["pk"], "auc": before["auc"], "tr_auc": before["tr_auc"],
                                       "test_base": before["base"], "client_overlap": before["overlap"]},
               "after_grouped_by_client": {"pk": after["pk"], "auc": after["auc"], "tr_auc": after["tr_auc"],
                                           "test_base": after["base"], "client_overlap": after["overlap"]},
               "groupkfold5": {"p50_mean": round(float(np.mean(p50s)), 3), "p50_sd": round(float(np.std(p50s)), 3),
                               "auc_mean": round(float(np.mean(aucs)), 3), "auc_sd": round(float(np.std(aucs)), 3)}},
           "section3": {
               "leak_trap": {"honest_p50": after["pk"][50], "honest_auc": after["auc"],
                             "leak_p50": l_p50, "leak_auc": l_auc},
               "feature_knowability": [{"feature": f, "what": w, "available_at_decision": a} for f, w, a in know],
               "client_overlap_grouped": ov_grp},
           "section4": {"claims_rewritten": [o for o, _ in rewrites]}}
json.dump(metrics, open(OUT / "w06_validation_metrics.json", "w"), indent=2)
print("metrics receipt written:", OUT / "w06_validation_metrics.json")


ORIG: P@50 0.86 / P@100 0.87 / AUC 0.739 on held-out clients - far above the baseline (0.44).
SAFE: Measured on the 13 held-out clients (base rate 0.429), the random forest scored P@50 0.86, P@100 0.87 and ROC-AUC 0.739 against the April-decline label, while the w04 rule scored P@50 0.44 on the same pages. Observed on this one client slice - not yet proven to generalize to other client mixes.

ORIG: momentum_last7 dominates the model's importance.
SAFE: In this sample, permutation importance measured on held-out pages ranked momentum_last7 first. That is directional evidence that recent-momentum loss co-occurs with the decline label - an association, not a proven cause, and not a lever you can pull to force retention.

ORIG: The two rankings share no top-50 pages, so both notions of 'worth reviewing' are real.
SAFE: On this slice the rule and the model share zero top-50 pages and both clear the rule's own P@K on the same split. That is an observation about two different ranking semanti

### Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed locally with the month cache)
- [x] No client names, URLs, or private queries anywhere — only hashes and aggregates
- [x] Base rates printed next to every P@K; grouped split proven via before/after overlap counts
- [x] Leakage hunt re-run on the final feature set, with a demonstrated leak trap
- [x] My claims re-written in safe language: observed, measured, directional, decision-support
- [x] Metrics receipt written to `work/outputs/w06_validation_metrics.json`
- [x] Committed to my repo under `work/notebooks/` — then submit repo URL on the card. Done.